# Financial Transactions Data Cleaning and Outlier Detection


In [1]:
import re
import numpy as np
import pandas as pd
from datetime import datetime

INPUT_FILE = "financial_transactions_uncleaned_4000_rows.xlsx"
OUTPUT_FILE = "financial_transactions_cleaned.xlsx"

df = pd.read_excel(INPUT_FILE, dtype=object)

original_rows, original_columns = df.shape
print("Original dataset shape:", df.shape)
display(df.head())

Original dataset shape: (3980, 15)


,transaction id,TRANSACTION_DATE,transaction amount,Currency,transaction_type,payment channel,Customer_Info,Transaction_Status,Location,Device_Type,Reference_Number,Notes,Unused_Column,Risk_Flag,Transaction_Time
0,TXN00001,2025-03-23,NaN,inr,transfer,mobile app,C2679 | Retail,NaN,madurai,ANDROID,REF809570@#,Regular transaction!!!,NaN,Review,01:39:00
1,TXN00002,03/01/2026,361.18,INR,Refund,Mobile App,C4611 | Premium,Pending,Madurai,Web,REF571029,Monthly transfer,NaN,Review,11:43 PM
2,TXN00003,2025-06-06,677.32,INR,Withdrawal,Mobile App,C6881 | BUSINESS,Success,Trichy,Android,REF988662,Monthly transfer,old-system,HIGH,06:52 PM
3,TXN00004,27/02/2025,1364.34,inr,Refund,Online,C4150 | Retail,Pending,Salem,Windows,REF838797,Regular transaction,NaN,Normal,09:07 AM
4,TXN00005,2025-10-04,334.51,inr,Withdrawal,Branch,C7065 | BUSINESS,Pending,Coimbatore,iOS,REF472528,Customer payment,old-system,Review,19:11:00


## Step 1: Removal of Empty Rows
**What:** delete rows where every cell is empty (cells that contain only spaces count as empty).
**Why:** empty rows carry no information and can break calculations.

In [2]:
def blank_to_nan(value):
    """Turn text that contains only spaces into a real missing value."""
    if isinstance(value, str) and value.strip() == "":
        return np.nan
    return value

df = df.apply(lambda column: column.map(blank_to_nan))

empty_rows_removed = int(df.isna().all(axis=1).sum())
df = df.dropna(how="all").copy()

print("Empty rows removed:", empty_rows_removed)
print("Shape after Step 1:", df.shape)

Empty rows removed: 0
Shape after Step 1: (3980, 15)


## Step 2: Removal of Duplicate Records
**What:** remove rows that are exact copies of an earlier row (every column identical).
**Why:** a repeated transaction would be counted twice in any analysis.
The first copy is kept, later copies are removed.

In [3]:
duplicates_removed = int(df.duplicated().sum())
df = df.drop_duplicates().copy()

print("Exact duplicate records removed:", duplicates_removed)
print("Shape after Step 2:", df.shape)

Exact duplicate records removed: 12
Shape after Step 2: (3968, 15)


## Step 3: Removal of Extra Spaces
**What:** remove leading, trailing and repeated spaces from the **column names** and from every **text value**.
**Note on the regular expression:** `\s+` means "one or more whitespace characters". In a raw string `r"..."` it must be written with a **single** backslash.

In [4]:
def clean_spaces(value):
    """Collapse repeated spaces into one and strip spaces at both ends."""
    if isinstance(value, str):
        value = re.sub(r"\s+", " ", value).strip()
        return value if value != "" else np.nan
    return value

# Column names
df.columns = [re.sub(r"\s+", " ", str(col)).strip() for col in df.columns]

# Text values
for col in df.columns:
    df[col] = df[col].map(clean_spaces)

print("Column names after Step 3:")
print(list(df.columns))

Column names after Step 3:
['transaction id', 'TRANSACTION_DATE', 'transaction amount', 'Currency', 'transaction_type', 'payment channel', 'Customer_Info', 'Transaction_Status', 'Location', 'Device_Type', 'Reference_Number', 'Notes', 'Unused_Column', 'Risk_Flag', 'Transaction_Time']


## Step 4: Text Case Standardization
**What:** (a) convert column names to `snake_case`; (b) give category text one consistent case (`Title Case`).
**Acronyms are protected:** POS, ATM and iOS must NOT become "Pos", "Atm" and "Ios", so a small dictionary is used for them.

In [5]:
# (a) column names -> lowercase snake_case
df.columns = [re.sub(r"[^a-z0-9]+", "_", col.lower()).strip("_") for col in df.columns]

# (b) category values -> Title Case, but keep acronyms correct
ACRONYMS = {"pos": "POS", "atm": "ATM", "ios": "iOS", "inr": "INR"}

def standardize_case(value):
    if isinstance(value, str):
        lower_value = value.lower()
        return ACRONYMS.get(lower_value, lower_value.title())
    return value

CATEGORY_COLUMNS = ["transaction_type", "payment_channel", "transaction_status",
                    "location", "device_type", "risk_flag"]

for col in CATEGORY_COLUMNS:
    df[col] = df[col].map(standardize_case)

df["currency"] = df["currency"].map(lambda v: v.upper() if isinstance(v, str) else v)

print("Column names:", list(df.columns))
print("Payment channels:", sorted(df["payment_channel"].dropna().unique()))
print("Device types    :", sorted(df["device_type"].dropna().unique()))
print("Risk flags      :", sorted(df["risk_flag"].dropna().unique()))

Column names: ['transaction_id', 'transaction_date', 'transaction_amount', 'currency', 'transaction_type', 'payment_channel', 'customer_info', 'transaction_status', 'location', 'device_type', 'reference_number', 'notes', 'unused_column', 'risk_flag', 'transaction_time']
Payment channels: ['ATM', 'Branch', 'Mobile App', 'Online', 'POS']
Device types    : ['Android', 'Web', 'Windows', 'iOS']
Risk flags      : ['High', 'Normal', 'Review']


## Step 5: Spelling Correction
**What:** fix spelling mistakes found in category values. The mistake found in this dataset is `Witdrawal` → `Withdrawal`.
The unique values are printed so you can see that no other misspelling remains.

In [6]:
spelling_fixes = {"Witdrawal": "Withdrawal"}

spelling_errors_fixed = int(df["transaction_type"].isin(spelling_fixes.keys()).sum())

for col in CATEGORY_COLUMNS:
    df[col] = df[col].replace(spelling_fixes)

print("Spelling errors corrected:", spelling_errors_fixed)
print("Transaction types now:", sorted(df["transaction_type"].dropna().unique()))

Spelling errors corrected: 21
Transaction types now: ['Deposit', 'Payment', 'Refund', 'Transfer', 'Withdrawal']


## Step 6: Date Format Standardization
**What:** the file contains four different date formats. All of them are converted to one real date type (`YYYY-MM-DD`).

| Format in the file | Example |
|---|---|
| `YYYY-MM-DD` | 2025-03-23 |
| `DD/MM/YYYY` | 27/02/2025 |
| `MM-DD-YYYY` | 03-25-2025 |
| `DD-Mon-YYYY` | 18-Oct-2025 |

Values that match none of these formats (for example `not-a-date`) are **not guessed**. They become blank and are flagged in `date_status`.
The original text is kept temporarily in `original_date` so the bad values can be shown in the `Invalid_Records` sheet.

In [7]:
DATE_FORMATS = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%d-%b-%Y"]

def parse_date(value):
    if pd.isna(value):
        return pd.NaT
    text = str(value).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.Timestamp(datetime.strptime(text, fmt))
        except ValueError:
            continue
    return pd.NaT

# Proof that the day/month order assumed above is right
# (a number greater than 12 can only be a day, never a month):
slash = df["transaction_date"].str.extract(r"^(\d{2})/(\d{2})/\d{4}$").dropna().astype(int)
dash = df["transaction_date"].str.extract(r"^(\d{2})-(\d{2})-\d{4}$").dropna().astype(int)
print("DD/MM/YYYY dates -> first number > 12:", int((slash[0] > 12).sum()),
      "| second number > 12:", int((slash[1] > 12).sum()))
print("MM-DD-YYYY dates -> first number > 12:", int((dash[0] > 12).sum()),
      "| second number > 12:", int((dash[1] > 12).sum()))

df["original_date"] = df["transaction_date"]          # temporary copy, removed before export
df["transaction_date"] = pd.to_datetime(df["transaction_date"].map(parse_date))

df["date_status"] = "Valid"
df.loc[df["original_date"].isna(), "date_status"] = "Missing"
df.loc[df["original_date"].notna() & df["transaction_date"].isna(), "date_status"] = "Invalid - Unrecognised format"

print()
print(df["date_status"].value_counts())
print("Earliest date:", df["transaction_date"].min().date(), "| Latest date:", df["transaction_date"].max().date())

DD/MM/YYYY dates -> first number > 12: 576 | second number > 12: 0
MM-DD-YYYY dates -> first number > 12: 0 | second number > 12: 607

date_status
Valid                            3965
Invalid - Unrecognised format       3
Name: count, dtype: int64
Earliest date: 2025-01-01 | Latest date: 2026-02-21


## Step 7: Time Format Standardization
**What:** the file mixes 24-hour times (`19:11:00`) and 12-hour times (`11:43 PM`). Both are converted to `HH:MM:SS`.
Then a combined column `transaction_datetime` (date + time) is created.

In [8]:
TIME_FORMATS = ["%H:%M:%S", "%I:%M %p", "%I:%M:%S %p", "%H:%M"]

def parse_time(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().upper()
    for fmt in TIME_FORMATS:
        try:
            return datetime.strptime(text, fmt).strftime("%H:%M:%S")
        except ValueError:
            continue
    return np.nan

df["transaction_time"] = df["transaction_time"].map(parse_time)

# date + time = combined datetime (blank if the date is blank)
df["transaction_datetime"] = df["transaction_date"] + pd.to_timedelta(df["transaction_time"], errors="coerce")

print("Times that could not be converted:", int(df["transaction_time"].isna().sum()))
display(df[["transaction_date", "transaction_time", "transaction_datetime"]].head())

Times that could not be converted: 0


,transaction_date,transaction_time,transaction_datetime
0,2025-03-23,01:39:00,2025-03-23 01:39:00
1,2026-01-03,23:43:00,2026-01-03 23:43:00
2,2025-06-06,18:52:00,2025-06-06 18:52:00
3,2025-02-27,09:07:00,2025-02-27 09:07:00
4,2025-10-04,19:11:00,2025-10-04 19:11:00


## Step 8: Splitting Columns
**What:** `customer_info` looks like `C4611 | Premium`. It is split into `customer_id` (`C4611`) and `customer_segment` (`Premium`).
The pattern also accepts extra spaces, lower/upper case and the 3 rows written without the `|` symbol (for example `C1234Retail`).

In [9]:
CUSTOMER_PATTERN = r"(?i)^\s*(C\d+)\s*\|?\s*(Retail|Premium|Business)\s*$"

customer_parts = df["customer_info"].str.extract(CUSTOMER_PATTERN)
df["customer_id"] = customer_parts[0].str.upper()
df["customer_segment"] = customer_parts[1].str.title()

rows_not_split = int((df["customer_id"].isna() | df["customer_segment"].isna()).sum())
print("Rows that could not be split:", rows_not_split)
print(df["customer_segment"].value_counts())
display(df[["customer_info", "customer_id", "customer_segment"]].head())

Rows that could not be split: 0
customer_segment
Retail      1623
Business    1589
Premium      756
Name: count, dtype: int64


,customer_info,customer_id,customer_segment
0,C2679 | Retail,C2679,Retail
1,C4611 | Premium,C4611,Premium
2,C6881 | BUSINESS,C6881,Business
3,C4150 | Retail,C4150,Retail
4,C7065 | BUSINESS,C7065,Business


## Step 9: Handling Missing Values
**What:** fill missing values only where a clear, honest default exists.

| Column | Action |
|---|---|
| `transaction_status` | missing → `Unknown` |
| `risk_flag` | missing → `Not Assigned` |
| `notes` | missing → `No Note` |
| `transaction_amount` | **left blank** (never guessed), flagged in Step 11 |
| `transaction_date` | **left blank** (never guessed), flagged in Step 6 |

In [10]:
missing_before = df[["transaction_status", "risk_flag", "notes", "transaction_amount", "transaction_date"]].isna().sum()

df["transaction_status"] = df["transaction_status"].fillna("Unknown")
df["risk_flag"] = df["risk_flag"].fillna("Not Assigned")
df["notes"] = df["notes"].fillna("No Note")

missing_after = df[["transaction_status", "risk_flag", "notes", "transaction_amount", "transaction_date"]].isna().sum()
print(pd.DataFrame({"missing_before": missing_before, "missing_after": missing_after}))

                    missing_before  missing_after
transaction_status              39              0
risk_flag                      824              0
notes                         1054              0
transaction_amount              68             68
transaction_date                 3              3


## Step 10: Data Type Conversion
**What:** amounts are stored as a mix of numbers and text such as `₹511.62`, `530.02 INR` or `Rs. 500`.
The currency words/symbols and commas are removed and the result is converted to a number.
Valid amounts containing `INR`, `Rs` or `₹` are kept. Text that is not a number (`abc`, `12O0`, `₹??`) becomes blank and is flagged in Step 11.

In [11]:
# keep the original text temporarily so bad values can be reported later
df["original_amount"] = df["transaction_amount"].map(lambda v: v if pd.isna(v) else str(v).strip())

amount_text = df["original_amount"].str.replace(r"(?i)\bINR\b|\bRs\b\.?|₹|,", "", regex=True).str.strip()
df["transaction_amount"] = pd.to_numeric(amount_text, errors="coerce").astype("float64")

print("Data type of transaction_amount:", df["transaction_amount"].dtype)
print("Amount cells containing text (symbols, words or letters):", int(df["original_amount"].map(lambda v: isinstance(v, str) and not re.fullmatch(r"-?\d+(\.\d+)?", v)).sum()))
print("Blank amounts after conversion:", int(df["transaction_amount"].isna().sum()))

Data type of transaction_amount: float64
Amount cells containing text (symbols, words or letters): 51
Blank amounts after conversion: 71


## Step 11: Identification of Invalid Values
**What:** every row gets an `amount_status`. The categories are kept **separate**:

| amount_status | Meaning |
|---|---|
| `Valid` | normal positive amount |
| `Missing` | the amount cell was empty |
| `Invalid - Non-numeric` | text that is not a number (`abc`, `12O0`, `₹??`) |
| `Placeholder` | dummy value `999999` or `-99999` |
| `Negative - Review` | a negative amount (other than the placeholder) |

These rows are **kept and flagged**, not deleted. They are listed in the `Invalid_Records` sheet, together with rows that have an invalid date.

In [12]:
PLACEHOLDER_VALUES = [999999.0, -99999.0]

amount = df["transaction_amount"]
is_missing = df["original_amount"].isna()
is_placeholder = amount.isin(PLACEHOLDER_VALUES)
is_non_numeric = df["original_amount"].notna() & amount.isna()
is_negative = (amount < 0) & ~is_placeholder

df["amount_status"] = "Valid"
df.loc[is_missing, "amount_status"] = "Missing"
df.loc[is_non_numeric, "amount_status"] = "Invalid - Non-numeric"
df.loc[is_placeholder, "amount_status"] = "Placeholder"
df.loc[is_negative, "amount_status"] = "Negative - Review"

print("Amount status:")
print(df["amount_status"].value_counts())
print()
print("Date status:")
print(df["date_status"].value_counts())

Amount status:
amount_status
Valid                    3891
Missing                    68
Negative - Review           3
Invalid - Non-numeric       3
Placeholder                 3
Name: count, dtype: int64

Date status:
date_status
Valid                            3965
Invalid - Unrecognised format       3
Name: count, dtype: int64


## Step 12: Removal of Unnecessary Columns
**What:** remove only columns that are not needed.
* `unused_column` – contains only the labels `old-system` / `legacy` and is not used in the analysis.
* `customer_info` – only removed **after** Step 8 split it successfully into two columns (safety check below), because it is now a duplicate.

No other column is removed.

In [13]:
columns_to_drop = ["unused_column"]

# customer_info is only dropped if every row was split correctly in Step 8
if df["customer_id"].notna().all() and df["customer_segment"].notna().all():
    columns_to_drop.append("customer_info")

df = df.drop(columns=[c for c in columns_to_drop if c in df.columns])

print("Columns removed:", columns_to_drop)
print("Shape after Step 12:", df.shape)

Columns removed:

 ['unused_column', 'customer_info']
Shape after Step 12: (3968, 20)


## Step 13: Removal of Unwanted Special Characters
**What:**
* `reference_number`: keep only letters, digits, `-` and `_` (so `REF809570@#` becomes `REF809570`).
* `notes`: remove symbols such as `! @ # $ %` but **keep** letters, digits, spaces and meaningful punctuation (`. , ' & ( ) / -`).
  If a note contained only symbols (for example `!!!`) nothing meaningful is left, so it becomes `No Note`.

In [14]:
reference_before = df["reference_number"].copy()
notes_before = df["notes"].copy()

df["reference_number"] = df["reference_number"].str.replace(r"[^A-Za-z0-9_-]", "", regex=True).str.upper()

df["notes"] = (
    df["notes"]
    .str.replace(r"[^A-Za-z0-9 .,'&()/-]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
df["notes"] = df["notes"].replace("", np.nan).fillna("No Note")

print("Reference numbers cleaned:", int((reference_before != df["reference_number"]).sum()))
print("Notes cleaned:", int((notes_before != df["notes"]).sum()))
print("Notes now:", sorted(df["notes"].unique()))

Reference numbers cleaned: 19
Notes cleaned: 18
Notes now: ['Customer payment', 'Monthly transfer', 'No Note', 'Regular transaction']


## Step 14: Category Standardization
**What:** make sure each category column only contains approved values.
* `currency`: `₹`, `RS` and `INR` all mean the Indian Rupee → standardized to `INR`.
* Every other category column is compared with its list of approved values. Any unexpected value would be printed here.

In [15]:
df["currency"] = df["currency"].replace({"RS": "INR", "RS.": "INR", "₹": "INR"})

VALID_CATEGORIES = {
    "currency": ["INR"],
    "transaction_type": ["Deposit", "Withdrawal", "Transfer", "Payment", "Refund"],
    "payment_channel": ["Branch", "Mobile App", "POS", "ATM", "Online"],
    "transaction_status": ["Success", "Pending", "Failed", "Unknown"],
    "location": ["Madurai", "Trichy", "Salem", "Coimbatore", "Chennai"],
    "device_type": ["Android", "iOS", "Web", "Windows"],
    "risk_flag": ["Normal", "Review", "High", "Not Assigned"],
    "customer_segment": ["Retail", "Premium", "Business"],
}

for col, allowed in VALID_CATEGORIES.items():
    unexpected = sorted(set(df[col].dropna()) - set(allowed))
    print(f"{col:20s} values: {sorted(df[col].dropna().unique())}  | unexpected: {unexpected}")

currency             values: ['INR']  | unexpected: []


transaction_type     values: ['Deposit', 'Payment', 'Refund', 'Transfer', 'Withdrawal']  | unexpected: []
payment_channel      values: ['ATM', 'Branch', 'Mobile App', 'Online', 'POS']  | unexpected: []
transaction_status   values: ['Failed', 'Pending', 'Success', 'Unknown']  | unexpected: []
location             values: ['Chennai', 'Coimbatore', 'Madurai', 'Salem', 'Trichy']  | unexpected: []
device_type          values: ['Android', 'Web', 'Windows', 'iOS']  | unexpected: []
risk_flag            values: ['High', 'Normal', 'Not Assigned', 'Review']  | unexpected: []
customer_segment     values: ['Business', 'Premium', 'Retail']  | unexpected: []


## Step 15: Resetting the Index
**What:** rows were removed in Steps 1–2, which leaves gaps in the row numbers. The index is renumbered 0, 1, 2, ...

In [16]:
df = df.reset_index(drop=True)

print("Index starts at", df.index.min(), "and ends at", df.index.max(), "| rows:", len(df))

Index starts at 0 and ends at 3967 | rows: 3968


## Step 16: Outlier Detection (IQR method)
**What:** an amount is a potential outlier if it is below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR`.
* Only rows with `amount_status = Valid` are used, so blank, text, placeholder and negative amounts cannot distort the limits.
* Outliers are **flagged, not deleted**. They stay in `Cleaned_Data` (column `is_amount_outlier`) and are also listed in the `Outliers_For_Review` sheet.

In [17]:
is_valid_amount = df["amount_status"] == "Valid"
valid_amounts = df.loc[is_valid_amount, "transaction_amount"]

q1 = valid_amounts.quantile(0.25)
q3 = valid_amounts.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

df["is_amount_outlier"] = is_valid_amount & (
    (df["transaction_amount"] < lower_fence) | (df["transaction_amount"] > upper_fence)
)
outliers_for_review = df[df["is_amount_outlier"]].copy()

print(f"Q1 = {q1:.2f} | Q3 = {q3:.2f} | IQR = {iqr:.2f}")
print(f"Lower fence = {lower_fence:.2f} | Upper fence = {upper_fence:.2f}")
print("Potential outliers (kept, not deleted):", len(outliers_for_review))
print("  below lower fence:", int((outliers_for_review["transaction_amount"] < lower_fence).sum()))
print("  above upper fence:", int((outliers_for_review["transaction_amount"] > upper_fence).sum()))

Q1 = 239.93 | Q3 = 694.48 | IQR = 454.55
Lower fence = -441.89 | Upper fence = 1376.29
Potential outliers (kept, not deleted): 258
  below lower fence: 0
  above upper fence: 258


## Step 17: Data Validation (exactly 16 checks)
Each check looks at the **final** cleaned data and shows `PASS` or `FAIL` with the real numbers behind it.
The checks are written independently of the cleaning code (for example the outlier check recalculates the quartiles with NumPy), so they can really catch mistakes.
If any check fails, the notebook stops before exporting.

In [18]:
# Columns that go into the final workbook (temporary helper columns original_date / original_amount are left out)
FINAL_COLUMNS = [
    "transaction_id", "transaction_date", "transaction_time", "transaction_datetime",
    "transaction_amount", "currency", "transaction_type", "payment_channel",
    "customer_id", "customer_segment", "transaction_status", "location", "device_type",
    "reference_number", "notes", "risk_flag", "amount_status", "date_status", "is_amount_outlier",
]
final_view = df[FINAL_COLUMNS]

TEXT_COLUMNS = ["transaction_id", "transaction_time", "currency", "transaction_type", "payment_channel",
                "customer_id", "customer_segment", "transaction_status", "location", "device_type",
                "reference_number", "notes", "risk_flag", "amount_status", "date_status"]

validation_checks = []

def add_check(name, passed, detail):
    validation_checks.append({
        "Check_No": len(validation_checks) + 1,
        "Validation_Check": name,
        "Result": "PASS" if bool(passed) else "FAIL",
        "Details": detail,
    })

amt = df["transaction_amount"]
status = df["amount_status"]

# 1
empty_rows = int(final_view.isna().all(axis=1).sum())
duplicate_rows = int(final_view.duplicated().sum())
add_check("No empty rows and no exact duplicate rows",
          empty_rows == 0 and duplicate_rows == 0,
          f"empty rows = {empty_rows}, duplicate rows = {duplicate_rows}")

# 2
ids_ok = df["transaction_id"].notna().all() and df["transaction_id"].is_unique \
         and df["transaction_id"].str.fullmatch(r"TXN\d+").all()
add_check("Transaction IDs are present, unique and in TXN##### format", ids_ok,
          f"{df['transaction_id'].nunique()} unique IDs for {len(df)} rows")

# 3
expected_rows = original_rows - empty_rows_removed - duplicates_removed
add_check("Row count = original rows - empty rows - duplicates (no rows lost by mistake)",
          len(df) == expected_rows,
          f"{original_rows} - {empty_rows_removed} - {duplicates_removed} = {expected_rows}; dataset has {len(df)}")

# 4
names_ok = all(re.fullmatch(r"[a-z0-9]+(_[a-z0-9]+)*", c) for c in final_view.columns) and final_view.columns.is_unique
add_check("Column names are lowercase snake_case and unique", names_ok, f"{final_view.shape[1]} columns")

# 5
space_problems = 0
for col in TEXT_COLUMNS:
    s = df[col].dropna().astype(str)
    space_problems += int(((s != s.str.strip()) | s.str.contains(r"\s{2,}")).sum())
add_check("No leading, trailing or repeated spaces in text columns", space_problems == 0,
          f"{space_problems} values with extra spaces")

# 6
add_check("transaction_amount is a numeric column", pd.api.types.is_float_dtype(amt), f"dtype = {amt.dtype}")

# 7
allowed_status = ["Valid", "Missing", "Invalid - Non-numeric", "Placeholder", "Negative - Review"]
status_ok = (status.isin(allowed_status).all()
             and (amt[status == "Valid"] > 0).all()
             and not amt[status == "Valid"].isin(PLACEHOLDER_VALUES).any()
             and amt[status == "Missing"].isna().all()
             and amt[status == "Invalid - Non-numeric"].isna().all()
             and amt[status == "Placeholder"].isin(PLACEHOLDER_VALUES).all()
             and (amt[status == "Negative - Review"] < 0).all())
add_check("amount_status is consistent with the amount (valid / missing / invalid / placeholder / negative)",
          status_ok, ", ".join(f"{k}={v}" for k, v in status.value_counts().items()))

# 8
blank_amounts = int(amt.isna().sum())
flagged_blank = int(status.isin(["Missing", "Invalid - Non-numeric"]).sum())
zero_amounts = int((amt == 0).sum())
# an original value like "530.02 INR", "Rs. 500" or "₹1,200.50" is a real amount and must NOT end up blank
REAL_AMOUNT_PATTERN = r"(?i)\s*(?:₹|rs\.?)?\s*-?[\d,]+(?:\.\d+)?\s*(?:inr)?\s*"
looks_like_real_amount = df["original_amount"].map(
    lambda v: isinstance(v, str) and re.fullmatch(REAL_AMOUNT_PATTERN, v) is not None)
lost_real_amounts = int((looks_like_real_amount & amt.isna()).sum())
add_check("Blank amounts were NOT made up, and no real amount (incl. INR / Rs / ₹ values) was lost",
          blank_amounts == flagged_blank and zero_amounts == 0 and lost_real_amounts == 0,
          f"blank amounts = {blank_amounts} (all flagged), zero amounts = {zero_amounts}, real amounts lost = {lost_real_amounts}")

# 9
today = pd.Timestamp.today().normalize()
valid_dates = df.loc[df["date_status"] == "Valid", "transaction_date"]
dates_ok = (pd.api.types.is_datetime64_any_dtype(df["transaction_date"])
            and df["date_status"].isin(["Valid", "Missing", "Invalid - Unrecognised format"]).all()
            and valid_dates.notna().all()
            and (valid_dates >= pd.Timestamp("2000-01-01")).all() and (valid_dates <= today).all()
            and df.loc[df["date_status"] != "Valid", "transaction_date"].isna().all())
add_check("Dates are real dates (not in the future); bad dates are blank and flagged", dates_ok,
          f"valid = {len(valid_dates)}, flagged = {int((df['date_status'] != 'Valid').sum())}")

# 10
time_ok = (df["transaction_time"].notna().all()
           and df["transaction_time"].str.fullmatch(r"\d{2}:\d{2}:\d{2}").all()
           and pd.to_datetime(df["transaction_time"], format="%H:%M:%S", errors="coerce").notna().all())
add_check("Every time is a valid HH:MM:SS value (no time values lost)", time_ok,
          f"{int(df['transaction_time'].notna().sum())} of {len(df)} times valid")

# 11
has_date = df["transaction_date"].notna()
dt = df["transaction_datetime"]
datetime_ok = ((dt[has_date].dt.strftime("%Y-%m-%d") == df.loc[has_date, "transaction_date"].dt.strftime("%Y-%m-%d")).all()
               and (dt[has_date].dt.strftime("%H:%M:%S") == df.loc[has_date, "transaction_time"]).all()
               and dt[~has_date].isna().all())
add_check("transaction_datetime matches transaction_date + transaction_time", datetime_ok,
          f"{int(dt.notna().sum())} datetimes created; {int((~has_date).sum())} blank because the date is invalid")

# 12
customer_ok = (df["customer_id"].notna().all() and df["customer_id"].str.fullmatch(r"C\d+").all()
               and df["customer_segment"].isin(VALID_CATEGORIES["customer_segment"]).all())
add_check("customer_id and customer_segment are filled and correctly split", customer_ok,
          f"{df['customer_id'].nunique()} unique customer IDs; segments: " + ", ".join(f"{k}={v}" for k, v in df['customer_segment'].value_counts().items()))

# 13
bad_category_values = 0
for col, allowed in VALID_CATEGORIES.items():
    bad_category_values += int((~df[col].isin(allowed)).sum())
add_check("Category columns only contain approved values (incl. POS, ATM, iOS, INR; no spelling errors)",
          bad_category_values == 0, f"{bad_category_values} unexpected values")

# 14
ref_ok = df["reference_number"].str.fullmatch(r"REF\d+").all()
notes_ok = df["notes"].notna().all() and df["notes"].str.fullmatch(r"[A-Za-z0-9 .,'&()/-]+").all()
add_check("Reference numbers and notes contain no unwanted special characters", ref_ok and notes_ok,
          f"reference format OK = {bool(ref_ok)}, notes clean = {bool(notes_ok)}")

# 15  (quartiles are recalculated with NumPy, independent of Step 16)
np_q1, np_q3 = np.percentile(df.loc[status == "Valid", "transaction_amount"], [25, 75])
np_low, np_high = np_q1 - 1.5 * (np_q3 - np_q1), np_q3 + 1.5 * (np_q3 - np_q1)
expected_outlier = (status == "Valid") & ((amt < np_low) | (amt > np_high))
outlier_ok = ((expected_outlier == df["is_amount_outlier"]).all()
              and len(outliers_for_review) == int(df["is_amount_outlier"].sum())
              and (df.loc[df["is_amount_outlier"], "amount_status"] == "Valid").all())
add_check("Outliers follow the IQR rule, use valid amounts only, and none were deleted", outlier_ok,
          f"{int(df['is_amount_outlier'].sum())} outliers flagged; limits {np_low:.2f} to {np_high:.2f}")

# 16
REQUIRED_COLUMNS = ["transaction_id", "transaction_time", "currency", "transaction_type", "payment_channel",
                    "customer_id", "customer_segment", "transaction_status", "location", "device_type",
                    "reference_number", "notes", "risk_flag", "amount_status", "date_status"]
missing_required = int(df[REQUIRED_COLUMNS].isna().sum().sum())
add_check("No unexpected missing values in required columns", missing_required == 0,
          f"{missing_required} missing values in {len(REQUIRED_COLUMNS)} required columns")

validation = pd.DataFrame(validation_checks)
assert len(validation) == 16, "There must be exactly 16 validation checks."

for _, row in validation.iterrows():
    print(f"[{row['Result']}] Check {row['Check_No']:>2}: {row['Validation_Check']}")
    print(f"           {row['Details']}")

passed_checks = int((validation["Result"] == "PASS").sum())
print()
print(f"Checks passed: {passed_checks} / {len(validation)}")
assert passed_checks == len(validation), "Some validation checks FAILED - fix them before exporting."

[PASS] Check  1: No empty rows and no exact duplicate rows


           empty rows = 0, duplicate rows = 0
[PASS] Check  2: Transaction IDs are present, unique and in TXN##### format
           3968 unique IDs for 3968 rows
[PASS] Check  3: Row count = original rows - empty rows - duplicates (no rows lost by mistake)
           3980 - 0 - 12 = 3968; dataset has 3968
[PASS] Check  4: Column names are lowercase snake_case and unique
           19 columns
[PASS] Check  5: No leading, trailing or repeated spaces in text columns
           0 values with extra spaces
[PASS] Check  6: transaction_amount is a numeric column
           dtype = float64
[PASS] Check  7: amount_status is consistent with the amount (valid / missing / invalid / placeholder / negative)
           Valid=3891, Missing=68, Negative - Review=3, Invalid - Non-numeric=3, Placeholder=3
[PASS] Check  8: Blank amounts were NOT made up, and no real amount (incl. INR / Rs / ₹ values) was lost
           blank amounts = 71 (all flagged), zero amounts = 0, real amounts lost = 0
[PASS] Che

### Step 17 (continued): Export the final workbook
Four sheets are created:

| Sheet | Content |
|---|---|
| `Cleaned_Data` | the full cleaned dataset (nothing deleted except empty rows and exact duplicates) |
| `Invalid_Records` | rows with a missing / non-numeric / placeholder / negative amount or an invalid date, with the reason and the original values |
| `Outliers_For_Review` | rows flagged as IQR outliers |
| `Validation` | the 16 checks with PASS / FAIL |

In [19]:
# ---- Invalid_Records ----
amount_issue = df["amount_status"].map({
    "Missing": "Missing amount",
    "Invalid - Non-numeric": "Non-numeric amount",
    "Placeholder": "Placeholder amount",
    "Negative - Review": "Negative amount",
})
date_issue = df["date_status"].map({
    "Missing": "Missing date",
    "Invalid - Unrecognised format": "Invalid date",
})
issue_found = pd.concat([amount_issue, date_issue], axis=1).apply(lambda row: "; ".join(row.dropna()), axis=1)
has_issue = issue_found != ""

invalid_records = df.loc[has_issue, FINAL_COLUMNS].copy()
invalid_records.insert(1, "issue_found", issue_found[has_issue])
invalid_records["original_date"] = df.loc[has_issue, "original_date"]
invalid_records["original_amount"] = df.loc[has_issue, "original_amount"]

# ---- other sheets ----
cleaned_data = df[FINAL_COLUMNS].copy()
outliers_sheet = df.loc[df["is_amount_outlier"], FINAL_COLUMNS].copy()

sheets = {
    "Cleaned_Data": cleaned_data,
    "Invalid_Records": invalid_records,
    "Outliers_For_Review": outliers_sheet,
    "Validation": validation,
}

from openpyxl.styles import Font

with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
    for sheet_name, sheet_df in sheets.items():
        sheet_df.to_excel(writer, sheet_name=sheet_name, index=False)
        ws = writer.sheets[sheet_name]
        ws.freeze_panes = "A2"
        for cell in ws[1]:
            cell.font = Font(bold=True)
        for idx, column_name in enumerate(sheet_df.columns, start=1):
            letter = ws.cell(row=1, column=idx).column_letter
            longest = max([len(str(column_name))] + [len(str(v)) for v in sheet_df[column_name].head(300)])
            ws.column_dimensions[letter].width = min(longest + 2, 60)
            if column_name == "transaction_date":
                for row_cells in ws.iter_rows(min_row=2, min_col=idx, max_col=idx):
                    row_cells[0].number_format = "yyyy-mm-dd"

print("Workbook saved as:", OUTPUT_FILE)

Workbook saved as: financial_transactions_cleaned.xlsx


### Confirm that the Excel file matches this notebook exactly
The saved workbook is opened again and compared with the data in memory: sheet names, number of rows and columns, and every value.

In [20]:
reloaded = pd.read_excel(OUTPUT_FILE, sheet_name=None)

def same_values(written, read_back):
    written = written.reset_index(drop=True)
    read_back = read_back.reset_index(drop=True)
    if pd.api.types.is_datetime64_any_dtype(written):
        fmt = "%Y-%m-%d %H:%M:%S"
        a = written.dt.strftime(fmt).fillna("blank")
        b = pd.to_datetime(read_back).dt.strftime(fmt).fillna("blank")
        return a.equals(b)
    if pd.api.types.is_numeric_dtype(written):
        return np.allclose(written.astype(float), read_back.astype(float), equal_nan=True)
    return written.astype(str).fillna("blank").equals(read_back.astype(str).fillna("blank"))

comparison = []
for sheet_name, sheet_df in sheets.items():
    excel_df = reloaded[sheet_name]
    shape_ok = excel_df.shape == sheet_df.shape and list(excel_df.columns) == list(sheet_df.columns)
    values_ok = shape_ok and all(same_values(sheet_df[c], excel_df[c]) for c in sheet_df.columns)
    comparison.append({"Sheet": sheet_name,
                       "Rows (notebook)": sheet_df.shape[0], "Rows (Excel)": excel_df.shape[0],
                       "Columns (notebook)": sheet_df.shape[1], "Columns (Excel)": excel_df.shape[1],
                       "All values identical": values_ok})

comparison = pd.DataFrame(comparison)
display(comparison)
assert comparison["All values identical"].all(), "Excel file does not match the notebook!"
print("The Excel workbook matches the notebook output exactly.")

,Sheet,Rows (notebook),Rows (Excel),Columns (notebook),Columns (Excel),All values identical
0,Cleaned_Data,3968,3968,19,19,True
1,Invalid_Records,80,80,22,22,True
2,Outliers_For_Review,258,258,19,19,True
3,Validation,16,16,4,4,True


The Excel workbook matches the notebook output exactly.


## Final Summary
All numbers below are calculated from the data when the notebook runs.

In [21]:
print("FINAL RESULTS")
print("-" * 60)
print(f"Original dataset            : {original_rows} rows x {original_columns} columns")
print(f"Empty rows removed          : {empty_rows_removed}")
print(f"Exact duplicates removed    : {duplicates_removed}")
print(f"Spelling errors corrected   : {spelling_errors_fixed}")
print(f"Cleaned dataset             : {cleaned_data.shape[0]} rows x {cleaned_data.shape[1]} columns")
print(f"Invalid records (flagged)   : {len(invalid_records)}")
print(f"Outliers (flagged, kept)    : {len(outliers_sheet)}")
print(f"Validation checks passed    : {passed_checks} / {len(validation)}")
print("Amount status counts        :", {k: int(v) for k, v in df["amount_status"].value_counts().items()})
print("Date status counts          :", {k: int(v) for k, v in df["date_status"].value_counts().items()})
print(f"Output file                 : {OUTPUT_FILE}")

FINAL RESULTS
------------------------------------------------------------
Original dataset            : 3980 rows x 15 columns
Empty rows removed          : 0
Exact duplicates removed    : 12
Spelling errors corrected   : 21
Cleaned dataset             : 3968 rows x 19 columns
Invalid records (flagged)   : 80
Outliers (flagged, kept)    : 258
Validation checks passed    : 16 / 16
Amount status counts        : {'Valid': 3891, 'Missing': 68, 'Negative - Review': 3, 'Invalid - Non-numeric': 3, 'Placeholder': 3}
Date status counts          : {'Valid': 3965, 'Invalid - Unrecognised format': 3}
Output file                 : financial_transactions_cleaned.xlsx
